# Is NGSolve getting Lazy?

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !uv pip install --system --pre ngsolve

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))

## Some people like the `SolvePDE` function

A left-over from pre-Python NGSolve times, but useful ...

Code from i-tutorials 1.3, going back to UM3

In [ ]:
fes = H1(mesh, order=2, dirichlet="left|right")  # Dirichlet boundaries in space
u,v = fes.TnT()
a = BilinearForm(grad(u)*grad(v)*dx)
f = LinearForm(1*v*dx).Assemble()
gfu = GridFunction(fes)
gfu.Set(sin(y), BND)      # interpolate Dirichlet boundary conditions
c = preconditioners.Local(a)   
c.Update()          # may or may not need update

Takes care of Dirichlet boundary conditions, static condensation, ...

In [ ]:
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c)
Draw(gfu);

## Version 2 
from UM5:

More verbose defining linear or non-linear problems:

In [ ]:
fes = H1(mesh, order=2, dirichlet="left|right")
u,v = fes.TnT()
a = BilinearForm(grad(u)*grad(v)*dx)
c = preconditioners.Local(a)   
f = LinearForm(1*v*dx).Assemble()
gfu = GridFunction(fes)

Solve(a*gfu==f, pre=c, \
      dirichlet=mesh.BoundaryCF({'left' : sin(y), 'right':-sin(y)}))
Draw (gfu);

## New attempt 

Only creator objects go into the `Solve` function

In [ ]:
u,v = H1(mesh, order=2).TnT()

lhs = grad(u)*grad(v)*dx
rhs = 1*v*dx

gfu = Solve(lhs==rhs, u[BND("left")]==sin(y), u[BND("right")]==-sin(y), \
            preconditioners.Local(), solvers.CGSolver.Creator())

Draw (gfu);

If we want to give arguments, provide it to the creator:

In [ ]:
gfu = Solve(lhs==rhs, u[BND("left|right")]==sin(y), \
            preconditioners.Local(blocktype="vertexpatch"), 
            solvers.CGSolver.Creator(maxiter=40, printrates=True),
            verbose=2) 
Draw (gfu);

## What are the new components?

### Region descriptors

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
fes = H1(mesh, order=1)
u,v = fes.TnT()
gfu = GridFunction(fes)

In [ ]:
gfu[BND("left")] = y
gfu[BND("top")] = 1-x
Draw (gfu);

In [ ]:
type (BND("left"))

In [ ]:
print (BND("left"))

In [ ]:
print (u[BND("left")])

In [ ]:
print (u[BND("left")]==3)

In [ ]:
reg = mesh[BND("left")]
print ("reg =", reg)
print ("reg.Mask() =", reg.Mask())

### Our three versions of setting boundary values of a `GridFunction`

The `Set` function performs element-wise $L_2$-projection, and then averaging coupling dofs.
It only needs basis-functions.

For historic reasons, the `Set` first zeros the complete `GridFunction`, thus consecutive calls to `Set` overwrites previous values.

In [ ]:
gfu = GridFunction(fes)
vals = mesh.BoundaryCF( { 'left' : y**2, 'right' : -y } )
gfu.Set(vals, definedon=mesh.Boundaries("left|right"))
Draw (gfu);

As we got degrees-of-freedom into NGSolve (aka DualShapes), we could implement canonical interpolation. This is now available for most spaces. The `Interpolate` function sets values on the given region, in leaves degrees of freedom not in the region unchanged. Note that the maximal value is now precisely as expected.

In [ ]:
gfu = GridFunction(fes)
gfu.Interpolate(y**2, definedon=mesh.Boundaries("right"))
gfu.Interpolate(-y, definedon=mesh.Boundaries("left"))
Draw (gfu);

The mesh enters via the `GridFunction`, and via the `Region` as well. There is some redundancy.

The new syntax using `GridFunction.__setitem__`is a compact notation for `Interpolate`:

In [ ]:
gfu = GridFunction(fes)
gfu[BND("left")] = -y
gfu[BND("right")] = y
Draw (gfu);

## Objects and Object creators

If we have a bilinear-from, we create a preconditioner, with some flags

In [ ]:
a = BilinearForm(grad(u)*grad(v)*dx)
c = preconditioners.Local(a, GS=True)

In [ ]:
type(c)

In [ ]:
print(c)

But if we don't have a `BilinearForm` object yet, we cannot create a preconditioner.

**First attempt:** Give a `Creator` static function to the preconditioner class, it provides a factory/builder/creator class

In [ ]:
creator = preconditioners.MultiGrid.Creator(cycle=2, smoother="block")
type(creator)

In [ ]:
pre = creator(a)
type(pre), print (pre)

**Second attempt:** Use the same class as class creator and final class object:

In [ ]:
pre_creator = preconditioners.MultiGrid(cycle=2, smoother="block")
type (pre_creator)

We did not give a `BilinearForm` object, so we have an *incomplete* object. However, it knows what type of (creator-)object it is, and stores arguments

In [ ]:
pre_creator.IsCreator()

finally, we can use it to create the real preconditioner object:

In [ ]:
pre = pre_creator.Create(a, cycle=2)
type (pre)

In [ ]:
pre.IsCreator()

The code of the new `Solve`-function (WIP):

In [ ]:
class VariationalEquationSolver:
    def __init__(self, equation, *args : DirichletBC | Preconditioner, verbose=0, **kwargs):
        self.bf = BilinearForm(equation.igls)

        self.dirichlet = [a for a in args if isinstance(a, DirichletBC)]
        self.fes = self.bf.space
        self.mesh = self.fes.mesh

        if self.dirichlet:
            self.dreg = sum((self.mesh[d.vbn] for d in self.dirichlet),
                            self.mesh[self.dirichlet[0].vbn])

        for a in args:
            if (verbose>=5): print ("got argument of type", type(a))

            if isinstance(a, Preconditioner):
                if a.IsCreator():
                    if verbose>=2: print ("PreconditionerCreator: ", type(a))
                    if self.dirichlet:
                        if verbose>=2: print ("Dirichlet: ", self.dreg.Mask())
                        self.pre = a.Create(self.bf, additional_dirichlet_constraints=self.dreg)
                    else:
                        self.pre = a.Create(self.bf)
                else:
                    self.pre = a

            if isinstance(a, LinearSolverCreator):
                self.linear_solver_creator = a
                        
            if isinstance(a, SparseFactorizationCreator):
                if verbose>=2: print ("SparseFactorizationCreator: ", type(a))                
                self.sparse_factorization_creator = a

                
        if pre := kwargs.get('pre'):
            self.pre=pre
                   
        
        
    def Solve(self):
        with TaskManager():
            gf = GridFunction(self.fes)
            for d in self.dirichlet:
                gf.ComponentFromProxy(d.proxy)[d.vbn] = d.val
            self.bf.AssembleLinearization(gf.vec)

            if hasattr(self, 'linear_solver_creator'):        
                inv = self.linear_solver_creator(self.bf.mat, self.pre)
            else:
                freedofs = self.fes.FreeDofs()
                if self.dirichlet:
                    freedofs = freedofs&(~self.fes.GetDofs(self.dreg))
                
                if hasattr(self, 'sparse_factorization_creator'):
                    inv = self.sparse_factorization_creator(self.bf.mat, freedofs)
                else:
                    inv = self.bf.mat.Inverse(freedofs)
            
            gf.vec.data -= inv * self.bf.Apply(gf.vec)
            return gf
        

def SolveVE (equation, *args, **kwargs):
     return VariationalEquationSolver(equation,*args,**kwargs).Solve()
